# ⚡ Nexus: LangGraph Multi-Agent Masterclass

Welcome to the **Nexus LangGraph Masterclass**. This notebook provides an interactive walkthrough of modern stateful multi-agent orchestration built with **LangGraph v1.2+**, **LangChain Core**, **LangSmith Observability**, and **Human-in-the-Loop** checkpoints.

---

### Key Capabilities Demonstrated:
1. **Stateful Graph Orchestration (`StateGraph`)**
2. **Reflection & Self-Correction Loops**
3. **Cross-Thread Long-Term Memory (`BaseStore`)**
4. **Time-Travel & State History Rewind (`get_state_history`)**
5. **Real-Time LangSmith Tracing**

In [ ]:
import os

from dotenv import load_dotenv

# Load environment variables
load_dotenv()
print(f"LangSmith Tracing Active: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")
print(f"Target Project: {os.getenv('LANGCHAIN_PROJECT', 'nexus-multi-agent-system')}")

## 1. Graph Compilation & Memory Initialization

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

from langgraph_multi_agent_system.graph import create_graph
from langgraph_multi_agent_system.memory_store import get_shared_store

# Instantiate graph with MemorySaver and shared Long-Term BaseStore
checkpointer = MemorySaver()
store = get_shared_store()
app = create_graph(
    checkpointer=checkpointer, store=store, interrupt_on_human_gate=False
)

print("Registered Nodes:", list(app.nodes.keys()))

## 2. Executing the Swarm Stream

In [ ]:
import uuid

thread_id = f"nb-{str(uuid.uuid4())[:8]}"
config = {"configurable": {"thread_id": thread_id}}

initial_state = {
    "task": "Analyze LangGraph StateGraph vs Linear Prompt Chains",
    "messages": [],
    "plan": [],
    "current_step": 0,
    "research_notes": [],
    "quantitative_data": [],
    "critique": {},
    "revision_count": 0,
    "max_revisions": 2,
    "next_agent": "",
    "requires_human_approval": False,
    "human_approved": None,
    "final_report": None,
    "metadata": {"thread_id": thread_id},
}

print(f"🚀 Launching Swarm (Thread: {thread_id})...")
for chunk in app.stream(initial_state, config=config):
    for node_name, state_update in chunk.items():
        msgs = state_update.get("messages", [])
        if msgs:
            print(f"● [{node_name.upper()}] {msgs[-1].content[:120]}...")

## 3. Inspecting the Cumulative State & Executive Report

In [ ]:
full_state = app.get_state(config).values
critique = full_state.get("critique", {})

print(f"Critic Score: {critique.get('score')}/100 | Passed: {critique.get('passed')}")
print(f"Triangulated Sources: {len(full_state.get('research_notes', []))}")
print("\n--- Executive Briefing ---\n")
print(full_state.get("final_report"))

## 4. Time-Travel & Checkpoint Rewind
LangGraph stores immutable checkpoints at every transition. We can inspect previous snapshots:

In [ ]:
from langgraph_multi_agent_system.time_travel import get_checkpoint_history

history = get_checkpoint_history(app, thread_id)
print(f"Found {len(history)} checkpoint steps in history:")
for idx, h in enumerate(reversed(history), 1):
    cid = str(h.get("checkpoint_id"))[:8]
    next_nodes = h.get("next_nodes") or ["END"]
    print(f" Step {idx}: [Checkpoint {cid}] -> Next: {next_nodes}")

## 5. Querying LangSmith Live Spans

In [ ]:
from langsmith import Client

client = Client()
runs = list(client.list_runs(project_name="nexus-multi-agent-system", limit=3))
print(f"Verified {len(runs)} recent traces in LangSmith project:")
for r in runs:
    print(f"- Run: {r.name} | Status: {r.status} | Latency: {r.latency or 0:.2f}s")